# Pemodelan dan Simulasi: Optimasi Campuran Larutan

Notebook ini menyelesaikan **UTS Soal 3: minimasi biaya campuran larutan netralisasi**. Isinya mencakup model program linear, eliminasi aljabar, enumerasi titik sudut eksak, pemeriksaan seluruh kendala, grafik daerah layak, pembuktian batas bawah biaya, evaluasi keluaran, dan GUI interaktif untuk mengecek campuran.

**Cara pakai:** buka/unggah di Google Colab lalu pilih **Runtime → Run all**. GUI berada pada bagian terakhir. Notebook membuat `titik_sudut.csv` dan `daerah_layak.png` di direktori kerja Colab.

Tautan GitHub yang diminta dosen tetap perlu diisi setelah script diunggah ke repositori kelompok.

## 1. Formulasi model matematika

Misalkan \(x\) adalah liter Larutan Basa dan \(y\) adalah liter Larutan Garam.

\[\min Z=20.000x+16.000y\]
\[4x+2y\ge24\quad	ext{(agen pengikat asam)}\]
\[2x+4y\ge24\quad	ext{(agen pengikat logam)}\]
\[x\ge0,\quad y\ge0.\]

Kebutuhan berbentuk minimum, sehingga kendala memakai tanda \(\ge\). Volume boleh pecahan menurut model, walau optimum soal ini berupa bilangan bulat. Model hanya menggunakan data yang diberikan pada soal.

In [ ]:
# Sel inti: solver eksak, evaluasi kendala, grafik, tabel titik sudut, dan ekspor
from fractions import Fraction as F
from itertools import combinations
from pathlib import Path
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink

CONSTRAINTS=[(F(4),F(2),F(24),'Kebutuhan asam'),
             (F(2),F(4),F(24),'Kebutuhan logam'),
             (F(1),F(0),F(0),'x nonnegatif'),
             (F(0),F(1),F(0),'y nonnegatif')]
COST=(F(20000),F(16000))

def feasible(x,y):
    return all(a*x+b*y>=rhs for a,b,rhs,_ in CONSTRAINTS)

def intersect(l1,l2):
    a,b,r,_=l1;c,d,s,_=l2;det=a*d-b*c
    if det==0:return None
    return (r*d-b*s)/det,(a*s-r*c)/det

def solve():
    candidates=sorted({pt for pair in combinations(CONSTRAINTS,2)
                       if (pt:=intersect(*pair)) is not None})
    vertices=[pt for pt in candidates if feasible(*pt)]
    if not vertices:raise ValueError('Tidak ditemukan titik sudut layak')
    optimum=min(vertices,key=lambda pt:COST[0]*pt[0]+COST[1]*pt[1])
    return candidates,vertices,optimum

candidates,vertices,(x_opt,y_opt)=solve()
def cost(x,y):return COST[0]*x+COST[1]*y
def rupiah(value):return f'Rp{int(value):,}'.replace(',','.')

display(Markdown("### Penyelesaian aljabar\n\nGaris batasnya 2x + y = 12 dan x + 2y = 12. Dari eliminasi diperoleh 3x = 12, sehingga x = 4 dan y = 4."))
all_rows=[]
for px,py in candidates:
    all_rows.append({'x Basa (L)':str(px),'y Garam (L)':str(py),
      'Agen asam':str(4*px+2*py),'Agen logam':str(2*px+4*py),
      'Biaya':rupiah(cost(px,py)),'Status':'Layak' if feasible(px,py) else 'Tidak layak'})
display(pd.DataFrame(all_rows))

# Verifikasi pemenuhan kebutuhan dan sertifikat batas bawah.
u,v=F(4000),F(2000)
assert (x_opt,y_opt)==(F(4),F(4))
assert 4*u+2*v==COST[0] and 2*u+4*v==COST[1]
assert 24*u+24*v==cost(x_opt,y_opt)==F(144000)
assert 4*x_opt+2*y_opt==24 and 2*x_opt+4*y_opt==24
print(f'Optimum: x={x_opt} L Basa, y={y_opt} L Garam')
print('Agen asam:',4*x_opt+2*y_opt,'unit; agen logam:',2*x_opt+4*y_opt,'unit')
print('Biaya minimum:',rupiah(cost(x_opt,y_opt)))
print('Bukti batas bawah: 20.000x + 16.000y >= 4.000(24) + 2.000(24) = Rp144.000.')
print('Solusi (4,4) mencapai batas bawah, jadi optimum global.')

# Visualisasi daerah layak dan garis isobiaya melalui optimum.
xx=np.linspace(0,14,500); yy=np.linspace(0,14,500); X,Y=np.meshgrid(xx,yy)
feas=(4*X+2*Y>=24)&(2*X+4*Y>=24)
fig,ax=plt.subplots(figsize=(8,6))
ax.contourf(X,Y,feas.astype(int),levels=[.5,1.5],colors=['#d9eaf5'],alpha=.9)
ax.plot(xx,12-2*xx,label='Asam: 4x + 2y = 24',color='#277da1')
ax.plot(xx,6-.5*xx,label='Logam: 2x + 4y = 24',color='#f9844a')
ax.plot(xx,9-1.25*xx,'--',color='#333',label='Biaya Rp144.000')
for px,py in vertices:
    ax.scatter(float(px),float(py),color='#222',zorder=4)
    ax.annotate(f'({px},{py})'+(' optimum' if (px,py)==(x_opt,y_opt) else ''),
                (float(px),float(py)),xytext=(7,8),textcoords='offset points')
ax.set(xlim=(0,14),ylim=(0,14),xlabel='Larutan Basa x (liter)',ylabel='Larutan Garam y (liter)',
       title='Daerah layak dan solusi biaya minimum')
ax.grid(alpha=.25);ax.legend();fig.tight_layout();fig.savefig('daerah_layak.png',dpi=180);plt.show()

pd.DataFrame([{'Basa (L)':str(a),'Garam (L)':str(b),'Biaya (Rp)':int(cost(a,b))} for a,b in vertices]).to_csv('titik_sudut.csv',index=False)
display(Markdown('File hasil dibuat: `titik_sudut.csv` dan `daerah_layak.png`.'))
for f in ['titik_sudut.csv','daerah_layak.png']:display(FileLink(f))

## 2. GUI pemeriksaan campuran

Atur volume Basa dan Garam. Tekan **Periksa campuran** untuk melihat jumlah kedua agen, status pemenuhan kebutuhan, biaya campuran, dan perbandingan dengan biaya optimum. Slider berfungsi sebagai alat uji input; solusi optimum model tetap dihitung dengan enumerasi titik sudut eksak.

In [ ]:
# GUI interaktif Google Colab / Jupyter
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except Exception:
    pass
basa=widgets.FloatSlider(value=4,min=0,max=15,step=.5,description='Basa (L):',readout_format='.1f')
garam=widgets.FloatSlider(value=4,min=0,max=15,step=.5,description='Garam (L):',readout_format='.1f')
button=widgets.Button(description='Periksa campuran',button_style='primary',icon='check')
out=widgets.Output()
def check_mix(_):
    x=F(str(basa.value));y=F(str(garam.value))
    acid=4*x+2*y;metal=2*x+4*y;z=cost(x,y)
    with out:
        clear_output(wait=True)
        status='✅ Memenuhi kedua kebutuhan' if feasible(x,y) else '❌ Belum memenuhi semua kebutuhan'
        display(Markdown(f"### {status}\n\n- Agen pengikat asam: **{acid} unit** (minimal 24)\n- Agen pengikat logam: **{metal} unit** (minimal 24)\n- Biaya campuran: **{rupiah(z)}**\n- Optimum model: **4 L Basa + 4 L Garam = Rp144.000**\n- Selisih dari biaya optimum: **{rupiah(z-cost(x_opt,y_opt))}**"))
button.on_click(check_mix)
display(widgets.VBox([basa,garam,button,out]))
button.click()

## 3. Interpretasi

Titik sudut layak beserta biayanya adalah (0,12) dengan Rp192.000, (4,4) dengan Rp144.000, dan (12,0) dengan Rp240.000. Optimum juga dapat dibuktikan tanpa hanya mengandalkan grafik: kalikan kendala asam dengan 4.000 dan kendala logam dengan 2.000, lalu jumlahkan. Setiap solusi layak harus memenuhi Z ≥ Rp144.000. Campuran (4,4) mencapai batas itu dan unik karena kedua persamaan kendala berpotongan pada satu titik. Penggunaan Garam saja memerlukan 12 liter dengan biaya Rp192.000, sedangkan Basa saja memerlukan 12 liter dengan biaya Rp240.000. Campuran optimum menghemat Rp48.000 (25%) dibanding Garam saja dan Rp96.000 (40%) dibanding Basa saja. Pada optimum, kedua agen terpenuhi tepat tanpa surplus.